# Adaptive FedAvg for Colab

This notebook is designed for Google Colab and supports multiple dataset formats:
- ImageFolder style (`train/class_a`, `train/class_b`, `test/...`)
- NIH-style metadata + image lists
- CSV-labeled datasets (`filepath`, `label`)

Update the configuration cell, then run all cells in order.

In [ ]:
# Colab setup
import os
import sys
import subprocess

def pip_install(packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + packages
    subprocess.check_call(cmd)

pip_install(['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'tqdm', 'Pillow'])

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Optional: Mount Google Drive
Uncomment and run if your dataset is in Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## Configuration
Set dataset mode and paths here.

In [ ]:
# =========================
# Dataset configuration
# =========================
# DATASET_MODE: 'auto' | 'imagefolder' | 'nih' | 'csv'
DATASET_MODE = 'auto'

# Root folder of uploaded dataset
DATASET_ROOT = '/content/dataset'

# CSV mode settings
CSV_PATH = '/content/dataset/labels.csv'
CSV_FILEPATH_COL = 'filepath'
CSV_LABEL_COL = 'label'
POSITIVE_LABELS = {'pneumonia', '1', 1, True}

# NIH mode settings (auto-detected when files exist)
NIH_CSV_NAME = 'Data_Entry_2017.csv'
NIH_TRAIN_LIST = 'train_val_list_NIH.txt'
NIH_TEST_LIST = 'test_list_NIH.txt'

# =========================
# Training configuration
# =========================
NUM_CLIENTS = 4
NUM_ROUNDS = 10
LOCAL_EPOCHS = 2
LOCAL_BATCH_SIZE = 16
IMAGE_SIZE = 128
LR = 5e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
SEED = 42

# Adaptive FedAvg
BETA_SIZE = 0.6
BETA_PERF = 0.4
PERF_TEMPERATURE = 5.0
MIN_CLIENT_WEIGHT = 1e-6

# Output folders
OUTPUT_DIR = '/content/adaptive_fedavg_outputs'
OUTPUT_HISTORY_DIR = '/content/adaptive_fedavg_outputs_history'
AUTO_ARCHIVE_OUTPUTS = True
PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_HISTORY_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

In [ ]:
import copy
import json
import math
import random
import shutil
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def archive_previous_outputs(output_dir, output_history_dir):
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(output_history_dir, exist_ok=True)
    existing_items = os.listdir(output_dir)
    if len(existing_items) == 0:
        print('No previous outputs to archive.')
        return None

    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    archive_dir = os.path.join(output_history_dir, f'run_{stamp}')
    os.makedirs(archive_dir, exist_ok=True)

    for item_name in existing_items:
        src = os.path.join(output_dir, item_name)
        dst = os.path.join(archive_dir, item_name)
        shutil.move(src, dst)

    print('Archived previous outputs to:', archive_dir)
    return archive_dir

set_seed(SEED)

class BinaryPathDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.targets = [label for _, label in samples]
        self.transform = transform
        self.classes = ['negative', 'positive']
        self.class_to_idx = {'negative': 0, 'positive': 1}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        with Image.open(path) as image:
            image = image.convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, label

def resolve_nih_images_root(data_dir):
    candidates = [
        os.path.join(data_dir, 'images-224', 'images-224'),
        os.path.join(data_dir, 'images-224'),
        os.path.join(data_dir, 'images'),
    ]
    for c in candidates:
        if os.path.isdir(c):
            return c
    return None

def is_nih_layout(data_dir):
    return all([
        os.path.isfile(os.path.join(data_dir, NIH_CSV_NAME)),
        os.path.isfile(os.path.join(data_dir, NIH_TRAIN_LIST)),
        os.path.isfile(os.path.join(data_dir, NIH_TEST_LIST)),
        resolve_nih_images_root(data_dir) is not None,
    ])

def is_imagefolder_layout(data_dir):
    train_root = os.path.join(data_dir, 'train')
    test_root = os.path.join(data_dir, 'test')
    return os.path.isdir(train_root) and os.path.isdir(test_root)

def build_nih_binary_datasets(data_dir, train_transform, eval_transform):
    csv_path = os.path.join(data_dir, NIH_CSV_NAME)
    train_list_path = os.path.join(data_dir, NIH_TRAIN_LIST)
    test_list_path = os.path.join(data_dir, NIH_TEST_LIST)
    images_root = resolve_nih_images_root(data_dir)

    metadata = pd.read_csv(csv_path, usecols=['Image Index', 'Finding Labels'])
    labels_by_image = dict(zip(metadata['Image Index'], metadata['Finding Labels']))

    def label_from_findings(findings):
        labels = [item.strip() for item in str(findings).split('|') if item.strip()]
        return 1 if 'Pneumonia' in labels else 0

    def read_split(list_path):
        with open(list_path, 'r', encoding='utf-8') as f:
            names = [line.strip() for line in f if line.strip()]
        samples = []
        for name in names:
            if name not in labels_by_image:
                continue
            p = os.path.join(images_root, name)
            if os.path.isfile(p):
                samples.append((p, label_from_findings(labels_by_image[name])))
        return samples

    train_samples = read_split(train_list_path)
    test_samples = read_split(test_list_path)
    if len(train_samples) == 0 or len(test_samples) == 0:
        raise RuntimeError('NIH split read produced empty train/test data.')

    train_ds = BinaryPathDataset(train_samples, transform=train_transform)
    test_ds = BinaryPathDataset(test_samples, transform=eval_transform)
    return train_ds, test_ds

def build_csv_binary_datasets(data_dir, csv_path, filepath_col, label_col, positive_labels, train_transform, eval_transform):
    df = pd.read_csv(csv_path)
    if filepath_col not in df.columns or label_col not in df.columns:
        raise ValueError(f'CSV must include columns: {filepath_col}, {label_col}')

    def norm_label(v):
        if isinstance(v, str):
            return v.strip().lower()
        return v

    positive_norm = set(norm_label(v) for v in positive_labels)
    samples = []
    for _, row in df.iterrows():
        raw_path = str(row[filepath_col])
        img_path = raw_path if os.path.isabs(raw_path) else os.path.join(data_dir, raw_path)
        if not os.path.isfile(img_path):
            continue
        label = 1 if norm_label(row[label_col]) in positive_norm else 0
        samples.append((img_path, label))

    if len(samples) == 0:
        raise RuntimeError('CSV mode found 0 usable samples.')

    random.shuffle(samples)
    split_idx = int(0.8 * len(samples))
    train_samples = samples[:split_idx]
    test_samples = samples[split_idx:]

    train_ds = BinaryPathDataset(train_samples, transform=train_transform)
    test_ds = BinaryPathDataset(test_samples, transform=eval_transform)
    return train_ds, test_ds

def build_datasets(data_dir, mode, train_transform, eval_transform):
    selected_mode = mode
    if mode == 'auto':
        if is_nih_layout(data_dir):
            selected_mode = 'nih'
        elif is_imagefolder_layout(data_dir):
            selected_mode = 'imagefolder'
        elif os.path.isfile(CSV_PATH):
            selected_mode = 'csv'
        else:
            raise RuntimeError('Could not auto-detect dataset format. Set DATASET_MODE manually.')

    if selected_mode == 'nih':
        train_ds, test_ds = build_nih_binary_datasets(data_dir, train_transform, eval_transform)
    elif selected_mode == 'imagefolder':
        train_root = os.path.join(data_dir, 'train')
        test_root = os.path.join(data_dir, 'test')
        train_ds = datasets.ImageFolder(train_root, transform=train_transform)
        test_ds = datasets.ImageFolder(test_root, transform=eval_transform)
    elif selected_mode == 'csv':
        train_ds, test_ds = build_csv_binary_datasets(
            data_dir, CSV_PATH, CSV_FILEPATH_COL, CSV_LABEL_COL, POSITIVE_LABELS, train_transform, eval_transform
        )
    else:
        raise ValueError(f'Unsupported DATASET_MODE: {selected_mode}')

    return train_ds, test_ds, selected_mode

In [ ]:
class PneumoniaCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.25),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

def iid_split(num_samples, num_clients):
    idx = list(range(num_samples))
    random.shuffle(idx)
    return [list(s) for s in np.array_split(idx, num_clients)]

def get_loader_from_indices(dataset, indices, batch_size, train_transform, eval_transform, shuffle_train=False):
    ds = copy.copy(dataset)
    if hasattr(dataset, 'samples'):
        ds.samples = [dataset.samples[i] for i in indices]
    if hasattr(dataset, 'targets'):
        ds.targets = [dataset.targets[i] for i in indices]
    ds.transform = train_transform if shuffle_train else eval_transform

    if shuffle_train:
        targets = ds.targets
        counts = {}
        for t in targets:
            counts[t] = counts.get(t, 0) + 1
        weights = [1.0 / counts[t] for t in targets]
        sampler = torch.utils.data.WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
        return DataLoader(ds, batch_size=batch_size, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)

    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

def local_train(model, dataloader, device, epochs=1, lr=1e-3, weight_decay=1e-4):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.95)
    criterion = nn.BCEWithLogitsLoss()

    running_loss = 0.0
    total_batches = 0
    pbar = tqdm(range(epochs), desc='Local epochs', leave=False)

    for _ in pbar:
        epoch_loss = 0.0
        epoch_batches = 0
        for imgs, labels in dataloader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.float().unsqueeze(1).to(device, non_blocking=True)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            v = float(loss.item())
            running_loss += v
            total_batches += 1
            epoch_loss += v
            epoch_batches += 1

        scheduler.step()
        pbar.set_postfix(avg_loss=f'{(epoch_loss / max(epoch_batches, 1)):.4f}')

    return model.state_dict(), running_loss / max(total_batches, 1)

@torch.no_grad()
def evaluate_model(model, dataloader, device):
    model.eval()
    ys, probs = [], []

    for imgs, labels in dataloader:
        imgs = imgs.to(device, non_blocking=True)
        logits = model(imgs)
        p = torch.sigmoid(logits).cpu().numpy().reshape(-1)
        probs.extend(p.tolist())
        ys.extend(labels.numpy().tolist())

    ys = np.array(ys)
    probs = np.array(probs)
    preds = (probs >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(ys, preds, labels=[0, 1]).ravel()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    sensitivity = recall
    balanced_accuracy = 0.5 * (recall + specificity)
    auc = roc_auc_score(ys, probs) if len(np.unique(ys)) > 1 else float('nan')

    return {
        'accuracy': float(accuracy_score(ys, preds)),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1_score(ys, preds, zero_division=0)),
        'auc': float(auc),
        'specificity': float(specificity),
        'sensitivity': float(sensitivity),
        'balanced_accuracy': float(balanced_accuracy),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
        'y_true': ys,
        'y_prob': probs,
    }

def adaptive_fedavg(local_weights, local_sizes, local_performances):
    total = sum(local_sizes)
    size_w = np.array([x / max(total, 1) for x in local_sizes], dtype=np.float64)

    perf = np.clip(np.array(local_performances, dtype=np.float64), 1e-6, 1.0)
    perf_scaled = np.power(perf, PERF_TEMPERATURE)
    perf_w = perf_scaled / np.sum(perf_scaled)

    w = BETA_SIZE * size_w + BETA_PERF * perf_w
    w = np.maximum(w, MIN_CLIENT_WEIGHT)
    w = w / np.sum(w)

    new_global = {}
    for k in local_weights[0].keys():
        if local_weights[0][k].dtype == torch.float32:
            new_global[k] = torch.zeros_like(local_weights[0][k])
        else:
            new_global[k] = local_weights[0][k].clone()

    for cw, lw in zip(w, local_weights):
        for k in lw.keys():
            if lw[k].dtype == torch.float32:
                new_global[k] += lw[k] * float(cw)

    return new_global, size_w.tolist(), perf_w.tolist(), w.tolist()

def compute_weight_drift(global_prev, global_new):
    sq_sum = 0.0
    numel = 0
    for k in global_prev.keys():
        if global_prev[k].dtype == torch.float32:
            diff = (global_new[k].cpu() - global_prev[k].cpu()).float()
            sq_sum += float(torch.sum(diff * diff).item())
            numel += diff.numel()
    if numel == 0:
        return 0.0
    return float(math.sqrt(sq_sum / numel))

def plot_confusion_matrix(tn, fp, fn, tp, save_path, title):
    matrix = np.array([[tn, fp], [fn, tp]])
    plt.figure(figsize=(6.5, 5.5))
    plt.imshow(matrix, cmap='Blues')
    plt.colorbar(label='Count')
    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(matrix[i, j]), ha='center', va='center', fontsize=12, fontweight='bold')
    plt.xticks([0, 1], ['Pred 0', 'Pred 1'])
    plt.yticks([0, 1], ['True 0', 'True 1'])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=180)
    plt.show()

In [ ]:
# Prepare transforms and dataset
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_train, test_dataset, mode_used = build_datasets(DATASET_ROOT, DATASET_MODE, train_transform, eval_transform)
print('Dataset mode used:', mode_used)
print('Train samples:', len(full_train), '| Test samples:', len(test_dataset))
print('Classes:', getattr(full_train, 'class_to_idx', 'n/a'))

client_splits = iid_split(len(full_train), NUM_CLIENTS)
clients = []
for split in client_splits:
    random.shuffle(split)
    cutoff = int(0.8 * len(split))
    clients.append({'train_idxs': split[:cutoff], 'val_idxs': split[cutoff:]})

print('Client dataset sizes (train, val):', [(len(c['train_idxs']), len(c['val_idxs'])) for c in clients])

test_loader = DataLoader(test_dataset, batch_size=LOCAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
# Train Adaptive FedAvg (aligned with main.py outputs)
if AUTO_ARCHIVE_OUTPUTS:
    archive_previous_outputs(OUTPUT_DIR, OUTPUT_HISTORY_DIR)
os.makedirs(PLOTS_DIR, exist_ok=True)

global_model = PneumoniaCNN().to(DEVICE)
global_weights = copy.deepcopy(global_model.state_dict())

global_round_rows = []
client_round_rows = []
weight_drift_rows = []

client_test_acc_history = {cid: [] for cid in range(NUM_CLIENTS)}
global_acc_history, global_f1_history, global_auc_history = [], [], []

for rnd in range(1, NUM_ROUNDS + 1):
    print(f'\n=== Adaptive FedAvg Round {rnd}/{NUM_ROUNDS} ===')
    selected_clients = list(range(NUM_CLIENTS))
    print('Selected clients:', selected_clients)

    for cid in range(NUM_CLIENTS):
        client_test_acc_history[cid].append(np.nan)

    local_weights, local_sizes, local_perf = [], [], []
    prev_global_cpu = {k: v.detach().cpu().clone() for k, v in global_weights.items()}

    for cid in selected_clients:
        train_loader = get_loader_from_indices(
            dataset=full_train,
            indices=clients[cid]['train_idxs'],
            batch_size=LOCAL_BATCH_SIZE,
            train_transform=train_transform,
            eval_transform=eval_transform,
            shuffle_train=True,
        )

        val_loader = get_loader_from_indices(
            dataset=full_train,
            indices=clients[cid]['val_idxs'],
            batch_size=LOCAL_BATCH_SIZE,
            train_transform=train_transform,
            eval_transform=eval_transform,
            shuffle_train=False,
        )

        local_model = PneumoniaCNN().to(DEVICE)
        local_model.load_state_dict(global_weights)

        updated_weights, train_loss = local_train(
            local_model, train_loader, DEVICE,
            epochs=LOCAL_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY
        )

        m_local = evaluate_model(local_model, val_loader, DEVICE)
        local_acc = m_local['accuracy']

        local_weights.append({k: v.detach().cpu().clone() for k, v in updated_weights.items()})
        local_sizes.append(len(clients[cid]['train_idxs']))
        local_perf.append(local_acc)

        client_test_acc_history[cid][-1] = local_acc

        client_round_rows.append({
            'round': rnd,
            'client': cid,
            'n_train_samples': len(clients[cid]['train_idxs']),
            'local_train_loss': train_loss,
            'local_test_accuracy': m_local['accuracy'],
            'local_test_precision': m_local['precision'],
            'local_test_recall': m_local['recall'],
            'local_test_f1': m_local['f1'],
            'local_test_auc': m_local['auc'],
            'local_test_specificity': m_local['specificity'],
            'local_test_sensitivity': m_local['sensitivity'],
            'local_test_balanced_accuracy': m_local['balanced_accuracy'],
            'size_weight': np.nan,
            'performance_weight': np.nan,
            'adaptive_weight': np.nan,
        })

        print(
            f'  Client {cid} -> TrainLoss: {train_loss:.4f}, '
            f'Acc: {m_local["accuracy"]:.4f}, Prec: {m_local["precision"]:.4f}, Rec: {m_local["recall"]:.4f}, '
            f'F1: {m_local["f1"]:.4f}, AUC: {m_local["auc"]:.4f}, Spec: {m_local["specificity"]:.4f}, '
            f'Sens: {m_local["sensitivity"]:.4f}, BalAcc: {m_local["balanced_accuracy"]:.4f}'
        )

    new_global_cpu, size_w, perf_w, adapt_w = adaptive_fedavg(local_weights, local_sizes, local_perf)

    selected_client_rows = [r for r in client_round_rows if r['round'] == rnd]
    for idx, row in enumerate(selected_client_rows):
        row['size_weight'] = size_w[idx]
        row['performance_weight'] = perf_w[idx]
        row['adaptive_weight'] = adapt_w[idx]

    global_weights = {k: v.to(DEVICE) for k, v in new_global_cpu.items()}
    global_model.load_state_dict(global_weights)

    drift_l2 = compute_weight_drift(prev_global_cpu, new_global_cpu)
    weight_drift_rows.append({'round': rnd, 'global_weight_drift_l2': drift_l2})

    m_global = evaluate_model(global_model, test_loader, DEVICE)
    global_acc_history.append(m_global['accuracy'])
    global_f1_history.append(m_global['f1'])
    global_auc_history.append(m_global['auc'])

    global_round_rows.append({
        'round': rnd,
        'global_accuracy': m_global['accuracy'],
        'global_precision': m_global['precision'],
        'global_recall': m_global['recall'],
        'global_f1': m_global['f1'],
        'global_auc': m_global['auc'],
        'global_specificity': m_global['specificity'],
        'global_sensitivity': m_global['sensitivity'],
        'global_balanced_accuracy': m_global['balanced_accuracy'],
        'global_tn': m_global['tn'],
        'global_fp': m_global['fp'],
        'global_fn': m_global['fn'],
        'global_tp': m_global['tp'],
        'mean_client_accuracy': float(np.nanmean([r['local_test_accuracy'] for r in selected_client_rows])),
        'std_client_accuracy': float(np.nanstd([r['local_test_accuracy'] for r in selected_client_rows])),
        'global_weight_drift_l2': drift_l2,
    })

    print(
        f'Global -> Acc: {m_global["accuracy"]:.4f}, Prec: {m_global["precision"]:.4f}, '
        f'Rec: {m_global["recall"]:.4f}, F1: {m_global["f1"]:.4f}, AUC: {m_global["auc"]:.4f}, '
        f'Spec: {m_global["specificity"]:.4f}, Sens: {m_global["sensitivity"]:.4f}, '
        f'BalAcc: {m_global["balanced_accuracy"]:.4f} | Drift(L2): {drift_l2:.6f}'
    )
    print('Adaptive weights:', [round(x, 4) for x in adapt_w])

print('\nTraining complete.')

In [ ]:
# Save outputs and plots (mirrors main.py artifacts)
dataset_name = os.path.basename(os.path.normpath(DATASET_ROOT)) or 'dataset'
dataset_tag = ''.join(ch if ch.isalnum() else '_' for ch in dataset_name.lower()).strip('_') or 'dataset'

def plot_path(stem):
    return os.path.join(PLOTS_DIR, f'{dataset_tag}_{stem}.png')

global_df = pd.DataFrame(global_round_rows)
client_round_df = pd.DataFrame(client_round_rows)
drift_df = pd.DataFrame(weight_drift_rows)

final_metrics = {
    'accuracy': float(global_acc_history[-1]) if len(global_acc_history) else float('nan'),
    'f1': float(global_f1_history[-1]) if len(global_f1_history) else float('nan'),
    'auc': float(global_auc_history[-1]) if len(global_auc_history) else float('nan'),
}

# Per-client final evaluation on local validation splits
per_client_results = []
for cid, client_info in enumerate(clients):
    val_idxs = client_info['val_idxs']
    if len(val_idxs) == 0:
        per_client_results.append({
            'client': cid, 'n_samples': 0,
            'accuracy': float('nan'), 'precision': float('nan'), 'recall': float('nan'),
            'f1': float('nan'), 'auc': float('nan'), 'specificity': float('nan'),
            'sensitivity': float('nan'), 'balanced_accuracy': float('nan'),
            'tn': np.nan, 'fp': np.nan, 'fn': np.nan, 'tp': np.nan,
        })
        continue

    val_loader = get_loader_from_indices(
        dataset=full_train,
        indices=val_idxs,
        batch_size=LOCAL_BATCH_SIZE,
        train_transform=train_transform,
        eval_transform=eval_transform,
        shuffle_train=False,
    )
    m = evaluate_model(global_model, val_loader, DEVICE)
    per_client_results.append({
        'client': cid,
        'n_samples': len(val_idxs),
        'accuracy': m['accuracy'],
        'precision': m['precision'],
        'recall': m['recall'],
        'f1': m['f1'],
        'auc': m['auc'],
        'specificity': m['specificity'],
        'sensitivity': m['sensitivity'],
        'balanced_accuracy': m['balanced_accuracy'],
        'tn': m['tn'],
        'fp': m['fp'],
        'fn': m['fn'],
        'tp': m['tp'],
    })

per_client_df = pd.DataFrame(per_client_results)

# Final global eval for ROC/confusion
final_test_eval = evaluate_model(global_model, test_loader, DEVICE)

# Save tabular outputs
global_df.to_csv(os.path.join(OUTPUT_DIR, 'global_round_metrics.csv'), index=False)
client_round_df.to_csv(os.path.join(OUTPUT_DIR, 'client_round_metrics.csv'), index=False)
drift_df.to_csv(os.path.join(OUTPUT_DIR, 'weight_drift.csv'), index=False)
per_client_df.to_csv(os.path.join(OUTPUT_DIR, 'per_client_results.csv'), index=False)

with open(os.path.join(OUTPUT_DIR, 'final_global_metrics.txt'), 'w', encoding='utf-8') as f:
    f.write(str(final_metrics))
with open(os.path.join(OUTPUT_DIR, 'final_global_metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(final_metrics, f, indent=2)

summary = {
    'timestamp': datetime.now().isoformat(),
    'config': {
        'dataset_mode': mode_used,
        'dataset_root': DATASET_ROOT,
        'dataset_name': dataset_name,
        'num_clients': NUM_CLIENTS,
        'num_rounds': NUM_ROUNDS,
        'local_epochs': LOCAL_EPOCHS,
        'local_batch_size': LOCAL_BATCH_SIZE,
        'image_size': IMAGE_SIZE,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'seed': SEED,
        'device': str(DEVICE),
        'adaptive': {
            'beta_size': BETA_SIZE,
            'beta_perf': BETA_PERF,
            'perf_temperature': PERF_TEMPERATURE,
        },
    },
    'final_metrics': final_metrics,
    'best_global_accuracy': float(np.nanmax(global_acc_history)) if len(global_acc_history) else float('nan'),
    'best_global_f1': float(np.nanmax(global_f1_history)) if len(global_f1_history) else float('nan'),
    'best_global_auc': float(np.nanmax(global_auc_history)) if len(global_auc_history) else float('nan'),
}
with open(os.path.join(OUTPUT_DIR, 'adaptive_fedavg_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

# Charts: global metrics
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(global_acc_history) + 1), global_acc_history, marker='o', linewidth=2, label='Global Accuracy')
plt.plot(range(1, len(global_f1_history) + 1), global_f1_history, marker='s', linewidth=2, label='Global F1')
plt.plot(range(1, len(global_auc_history) + 1), global_auc_history, marker='^', linewidth=2, label='Global AUC')
plt.plot(range(1, len(global_round_rows) + 1), [row['global_balanced_accuracy'] for row in global_round_rows], marker='d', linewidth=2, label='Global Balanced Acc')
plt.xlabel('Communication Round')
plt.ylabel('Metric Value')
plt.title(f'Global Metrics vs Rounds (Adaptive FedAvg) - {dataset_name}')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(plot_path('global_metrics_vs_rounds'), dpi=150)
plt.show()

# Charts: weight drift
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(weight_drift_rows) + 1), [row['global_weight_drift_l2'] for row in weight_drift_rows], marker='o', linewidth=2, color='tab:purple')
plt.xlabel('Communication Round')
plt.ylabel('L2 Drift')
plt.title(f'Global Weight Drift vs Rounds - {dataset_name}')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(plot_path('weight_drift_vs_rounds'), dpi=150)
plt.show()

# Charts: client accuracy trajectories (smoothed)
plt.figure(figsize=(12, 7))
colors = plt.cm.Set2(np.linspace(0, 1, NUM_CLIENTS))
for cid in range(NUM_CLIENTS):
    rounds = list(range(1, len(client_test_acc_history[cid]) + 1))
    accs = client_test_acc_history[cid]
    smoothed = []
    ema = None
    for acc in accs:
        if not np.isnan(acc):
            ema = acc if ema is None else 0.3 * acc + 0.7 * ema
            smoothed.append(ema)
        else:
            smoothed.append(np.nan)
    plt.plot(rounds, smoothed, marker='o', linewidth=2.5, markersize=6, color=colors[cid], label=f'Client {cid}')
plt.xlabel('Federated Round')
plt.ylabel('Shared Test Accuracy')
plt.title(f'Client-wise Accuracy Trajectories (Smoothed) - {dataset_name}')
plt.grid(True, alpha=0.4, linestyle='--')
plt.legend(loc='best', ncol=2)
plt.xticks(range(1, NUM_ROUNDS + 1))
plt.ylim(0.0, 1.0)
plt.tight_layout()
plt.savefig(plot_path('client_accuracy_over_rounds'), dpi=150)
plt.show()

# Charts: client convergence
plt.figure(figsize=(10, 6))
for cid in range(NUM_CLIENTS):
    client_series = [row['local_test_accuracy'] for row in client_round_rows if row['client'] == cid]
    plt.plot(range(1, len(client_series) + 1), client_series, linestyle='--', alpha=0.6, label=f'Client {cid} local')
plt.plot(range(1, len(global_acc_history) + 1), global_acc_history, color='black', linewidth=3, label='Global aggregated')
plt.xlabel('Round')
plt.ylabel('Accuracy')
plt.title(f'Client Convergence Under Adaptive Aggregation - {dataset_name}')
plt.grid(True, alpha=0.3)
plt.legend(ncol=2)
plt.tight_layout()
plt.savefig(plot_path('client_convergence_effect'), dpi=150)
plt.show()

# Charts: final per-client metrics
plt.figure(figsize=(8, 5))
plt.bar(per_client_df['client'].astype(str), per_client_df['accuracy'], label='Accuracy', alpha=0.85)
plt.plot(per_client_df['client'].astype(str), per_client_df['f1'], marker='o', linewidth=2, label='F1')
plt.plot(per_client_df['client'].astype(str), per_client_df['auc'], marker='s', linewidth=2, label='AUC')
plt.xlabel('Client ID')
plt.ylabel('Score')
plt.title(f'Final Per-Client Validation Metrics - {dataset_name}')
plt.ylim(0, 1.05)
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(plot_path('final_metrics_per_client'), dpi=150)
plt.show()

# Charts: ROC + confusion matrix
if len(np.unique(final_test_eval['y_true'])) > 1:
    fpr, tpr, _ = roc_curve(final_test_eval['y_true'], final_test_eval['y_prob'])
    auc_val = roc_auc_score(final_test_eval['y_true'], final_test_eval['y_prob'])
    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f'ROC (AUC={auc_val:.4f})')
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - Final Global Model ({dataset_name})')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(plot_path('roc_curve_global'), dpi=150)
    plt.show()

plot_confusion_matrix(
    final_test_eval['tn'],
    final_test_eval['fp'],
    final_test_eval['fn'],
    final_test_eval['tp'],
    save_path=plot_path('confusion_matrix_global'),
    title=f'Global Model Confusion Matrix ({dataset_name})',
)

print('Saved outputs to:', OUTPUT_DIR)
print('Saved plots to:', PLOTS_DIR)

## Notes for Other Datasets
- For non-medical datasets, update `POSITIVE_LABELS` and label mapping logic.
- For multiclass tasks, replace binary loss/metrics with multiclass versions.
- If your dataset has no predefined test split, use CSV mode or split train data manually.